# SA-Jev · 公开 LightRAG 索引与数据准备
默认使用三套公开 N500 图，不对训练段落重新付费建图。产物保存到 MyDrive/SA-Jev/public_lightrag_v1/pilot 或 full；旧 full 目录保留。full：8,000+4,000 道训练题；HotpotQA/MuSiQue 每库 300 测试、100 验证、100 扰动保留，2Wiki 500 测试。非训练题全部来自公开图配套清单，不表示 benchmark 全测试集。CPU 可运行，下载约 561 MB 索引和小型 MiniLM；不下载 OpenJev-4B。首次先跑 pilot；成功结束后默认刷新 Drive 并断开。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Use 'full' after the pilot; separate Drive directories.
AUTO_DISCONNECT = True  # Release Colab after successful preparation and Drive flush.
from uuid import uuid4
PREPARATION_RUN_ID = uuid4().hex
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


: 

: 

In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [ ]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [ ]:
import torch
from pathlib import Path
from tempfile import mkdtemp
from sa_jev.smoke import run

smoke_parent = ROOT / "smoke"
smoke_parent.mkdir(parents=True, exist_ok=True)

smoke_dir = Path(mkdtemp(prefix="run_", dir=smoke_parent))
print("本次测试目录：", smoke_dir)

summary = run(
    smoke_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
summary

本次测试目录： /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/smoke/run_4jgbai8v
{'step': 1, 'epoch': 0, 'rank': 1.7917581796646118, 'suf': 0.6931471526622772, 'rel': 0.6931471526622772, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 1.018705200000113, 'eta_seconds': 1.018705200000113, 'grad_norm': 0.0001167683003586717, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 10, 'task': 2.6235347628593444, 'total_loss': 2.6235347628593444}
{'step': 2, 'epoch': 0, 'rank': 1.6094295978546143, 'suf': 0.693147212266922, 'rel': 0.6931336522102356, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 2.2358591980009805, 'eta_seconds': 0.0, 'grad_norm': 0.0002577165432740003, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 9, 'task': 2.4412035405635835, 'total_loss': 2.4412035405635835, 'validation': {'rank': 1.7005923390388489, 'su

{'test_only': True,
 'output_directory': '/content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/smoke/run_4jgbai8v',
 'model': 'tiny causal encoder + actual upstream OpenJev DecisionHead/OptionSetInteractor',
 'data': 'synthetic fixtures',
 'device': 'cpu',
 'train_states': 6,
 'validation_states': 3,
 'preference_pairs': 6,
 'stage1_optimizer_steps': 2,
 'stage2_optimizer_steps': 2,
 'checkpoint_roundtrip': True,
 'evaluation_rows': 1,
 'evaluation_errors': 0,
 'peak_reserved_gib': 0.0,
 'not_verified': ['trained OpenJev-4B weights',
  'A100 80GB peak',
  'live LightRAG extraction',
  'official Jev API',
  'scientific performance']}

公开图的向量编码器为 all-MiniLM-L6-v2（384 维、256 token），与 BGE 不兼容。本地下载固定版本，仅用于查询和兼容性检查，不调用 embedding API。源发布未记录编码器 revision，加载图后用已存向量进行兼容性校验。

In [ ]:
from sa_jev.embeddings import check_embeddings
embedding_check = await check_embeddings(config['retrieval'], ROOT/'embedding_check.json')
embedding_check

Loading local embedding: sentence-transformers/all-MiniLM-L6-v2@1110a243fdf4706b3f48f1d95db1a4f5529b4d41, device=cpu
Checking pinned sentence-transformers/all-MiniLM-L6-v2 weight: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/model.safetensors
Embedding weight cache is invalid: expected 90,868,376 bytes, got 79. Re-downloading the same revision once.
Preserved invalid cache entry: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/model.safetensors.invalid-86682ba9


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

sentence-transformers/all-MiniLM-L6-v2 weight verified: 90,868,376 bytes, SHA-256 matches.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding ready in 6.7s: dim=384, batch=16, max_length=256, normalized=True
Embedding check: (2, 384), finite=True, 0.12s, GPU peak reserved=0.000 GiB


{'backend': 'local',
 'model': 'sentence-transformers/all-MiniLM-L6-v2',
 'revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41',
 'dimension': 384,
 'device': 'cpu',
 'load_seconds': 16.011506373000884,
 'encode_seconds': 0.11617254299926572,
 'texts_per_second': 25.82365783297833,
 'document_norms': [1.0, 1.0],
 'toy_top1_correct': True,
 'peak_gpu_reserved_gib': 0,
 'finite_vectors': True,
 'test_only': True}

In [ ]:
# Retired one-time repair: zip_path_patch
# The current project already includes this fix. Original code is preserved in cell metadata.
print("Skipped obsolete repair; continuing with the current Git code.")


Skipped obsolete repair; continuing with the current Git code.


下载并校验公开 ZIP，按重复条目分别导入三套图；2Wiki 使用包内完整冲突副本，详见 public_manifest.json。复用旧 full/raw 下载（若完整），否则下载原始训练源。仅采样训练问题的段落用于 BM25 难例，不调用建图 API。公开配套题按问题族分为测试、验证和独立扰动保留题；不再从配套清单外抽题。旧划分首次运行时归档其派生数据与完成标记，保留图、原始下载和逐题候选缓存。少量缺失支持仍如实报告，不注入金标。

In [ ]:
import importlib
from sa_jev import embeddings, retrieval, public_indexes, preparation, experiments
for module in (embeddings, retrieval, public_indexes, preparation, experiments):
    importlib.reload(module)
config = experiments.load_config(CONFIG_PATH)
from sa_jev.public_indexes import download_public_assets, import_public_indexes, prepare_public_data
from sa_jev.preparation import download_sources
from sa_jev.io import read_json
archive = download_public_assets(PUBLIC_ASSETS)
index_manifest = import_public_indexes(archive, ROOT)
raw = ROOT/'raw'
legacy = DRIVE/'full/raw'
if MODE == 'full' and (legacy/'download_manifest.json').exists() and read_json(legacy/'download_manifest.json').get('formal'):
    raw = legacy
    print('Reusing existing source downloads:', raw)
download_sources(raw, limit=64 if MODE=='pilot' else None)
manifest = prepare_public_data(ROOT, PUBLIC_ASSETS, raw, config, mode=MODE)
manifest

Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/lightrag_indexes_N500.zip
Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/hotpotqa_queries.json
Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/musique_queries.json
Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/2wiki_queries.json
Reusing imported public indexes; no document extraction
Reusing existing source downloads: /content/drive/MyDrive/SA-Jev/full/raw
[data] Reusing /content/drive/MyDrive/SA-Jev/full/raw/musique_ans_train.jsonl
[data] Reusing /content/drive/MyDrive/SA-Jev/full/raw/musique_ans_dev.jsonl
[data] Reusing /content/drive/MyDrive/SA-Jev/full/raw/hotpot_train.jsonl
[data] Reusing /content/drive/MyDrive/SA-Jev/full/raw/hotpot_validation.jsonl
[data] All source downloads already complete; no network request needed.
Archived previous question protocol: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/protocol_archive/f336357fdbadb520. Existing graphs and candidate pool

{'preparation_binding': {'protocol': 'public_lightrag_n500_v1',
  'mode': 'full',
  'counts': {'hotpotqa': 8000, 'musique': 4000},
  'seed': 17,
  'validation_per_dataset': 100,
  'heldout_per_dataset': 100,
  'question_protocol': 'published_family_split_v2',
  'queries': {'hotpotqa': '3ad9c0bcbf93f41d7004ca6007049c904d2605046b314a2f7ecfb379c64cba6d',
   'musique': '98ed4e21d3076532f6388d42320fb809599c63a0d8dffca8ece5e41922be6b46',
   '2wiki': '895cba294064df0c3302c76847b1fc08d99b5619f7663dfaa3b65cd780f1cac4'},
  'raw': {'hotpotqa': '9ad5804dc246b20ceccc238f4873789493184b3f1a29f56bba242feb4e452aea',
   'musique': '0ab34edc31d1db89b392c202a1a1ae84a8b6198818324ffdc10956a940ad5d6d'}},
 'protocol': 'public_lightrag_n500_v1',
 'question_protocol': 'published_family_split_v2',
 'question_fingerprint': '66fc48b179a68558ffcf826467524d029ef74722e18f32b9b2a281b6d5adfc5d',
 'corpus_fingerprint': '205861e860b280873c2a4e4a71564f1b775196bd3bc6e7bebf96c0160687f451',
 'questions': 13500,
 'corpus_docu

In [ ]:
# Retired one-time repair: forced_minilm_redownload
# The current project already includes this fix. Original code is preserved in cell metadata.
print("Skipped obsolete repair; continuing with the current Git code.")


Skipped obsolete repair; continuing with the current Git code.


In [9]:
# Retired one-time repair: manual_source_zip_overlay
# The current project already includes this fix. Original code is preserved in cell metadata.
print("Skipped obsolete repair; continuing with the current Git code.")


Skipped obsolete repair; continuing with the current Git code.


在导入的三个图上执行 LightRAG 原生 mix（图＋向量）召回，缓存最大 K=256，再切片得到主实验 K=32。保留已发布图，不重新抽取文档；查询关键词提取仍会调用 .env 中的 DeepSeek。完整协议有 1,500 个查询（1,100 测试、200 验证、200 独立保留），pilot 18 个。先检查 public_source_coverage.json，再执行检索；同题、同图、同配置的旧缓存可复用。初始缓存完成后，其余 notebook 复用候选。检查 public_pool_coverage.csv；缺失支持、未知标签仍掩蔽。

In [10]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

Public source coverage: {'dataset': 'hotpotqa', 'split': 'validation', 'questions': 100, 'all_support': 99, 'no_support': 0}
Public source coverage: {'dataset': 'hotpotqa', 'split': 'heldout', 'questions': 100, 'all_support': 100, 'no_support': 0}
Public source coverage: {'dataset': 'hotpotqa', 'split': 'test', 'questions': 300, 'all_support': 297, 'no_support': 0}
Public source coverage: {'dataset': 'musique', 'split': 'validation', 'questions': 100, 'all_support': 100, 'no_support': 0}
Public source coverage: {'dataset': 'musique', 'split': 'heldout', 'questions': 100, 'all_support': 100, 'no_support': 0}
Public source coverage: {'dataset': 'musique', 'split': 'test', 'questions': 300, 'all_support': 300, 'no_support': 0}
Public source coverage: {'dataset': '2wiki', 'split': 'test', 'questions': 500, 'all_support': 498, 'no_support': 0}
Public pool 1/1500 hotpotqa; reused; K=256; ETA=125s; extraction calls=0
Public pool 2/1500 hotpotqa; reused; K=256; ETA=85s; extraction calls=0
Publ

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

sentence-transformers/all-MiniLM-L6-v2 weight verified: 90,868,376 bytes, SHA-256 matches.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding ready in 5.9s: dim=384, batch=16, max_length=256, normalized=True
Stored MiniLM compatibility: min cosine=1.000000


... (1,999 outputs left out)

Public pool 1499/1500 2wiki; new query; K=256; ETA=6s; extraction calls=0


INFO:  == LLM cache == saving: mix:keywords:0dcb2fd7a44fc745c3beb443d93cf3c6
INFO: Query nodes: Así En El Cielo Como En La Tierra, Buck And The Preacher (top_k:256, cosine:0.2)
INFO: Local query: 256 entites, 405 relations
INFO: Query edges: film director birth date comparison, director born later (top_k:256, cosine:0.2)
INFO: Global query: 405 entites, 256 relations
INFO: Naive query: 256 chunks (chunk_top_k:256 cosine:0.2)
INFO: Raw search results: 658 entities, 659 relations, 256 vector chunks
INFO: After truncation: 361 entities, 489 relations
INFO: Selecting 390 from 390 entity-related chunks by vector similarity
INFO: Find 89 additional chunks in 89 relations (deduplicated 178)
INFO: Selecting 89 from 89 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 735 -> 604 (deduplicated 131)
INFO: Final context: 361 entities, 489 relations, 256 chunks
INFO: Final chunks S+F/O: E3/1 R1/1 E2/2 R2/2 E6/3 R1/3 E8/4 R1/4 E2/5 R1/5 E1/6 R1/6 E2/7 R1/7 E2/8 R1/8 E2/9 

Public pool 1500/1500 2wiki; new query; K=256; ETA=0s; extraction calls=0


INFO: Successfully finalized 12 storages


In [5]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

: 

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [ ]:
cli('overlap')
cli('review-packets')

In [ ]:
from sa_jev.preparation import finish_preparation
completion = finish_preparation(ROOT, config, mode=MODE, run_id=PREPARATION_RUN_ID)
completion

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。200 道保留题不等于 600 个合格状态包：实际草稿数量和独立题数见 perturbations_draft.manifest.json；只报告核验通过的实际数量。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。

结束后自动断开：只有本次运行已成功完成才执行。先刷新并卸载 Drive，再释放 Colab 运行时。Drive 刷新失败或流程失败时不主动释放；重新连接需重装环境并挂载 Drive，已保存数据保留。本地运行会跳过此步骤。

In [ ]:
from sa_jev.environment import disconnect_after_preparation
disconnect_after_preparation(ROOT, run_id=PREPARATION_RUN_ID, enabled=AUTO_DISCONNECT)